# 2.2 · Prophet — baseline aditivo

**Tiempo estimado:** 30 min.

**Objetivos.**

1. Ajustar Prophet al caudal mensual del Genil con lluvia como regresor.
2. Inspeccionar componentes (tendencia + estacionalidad anual + lluvia).
3. Comparar contra SARIMAX/ETS de 01.

In [ ]:
import logging

logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from prophet import Prophet

from cst import datos as ud

plt.rcParams.update({"figure.figsize": (10, 3.4), "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
caudal_m = (
    ud.cargar_caudal_genil(source="CEDEX")
    .resample("MS")
    .mean()
    .loc["1995":"2020"]
    .interpolate("linear", limit=2)
    .dropna()
)
lluvia_m = (
    ud.cargar_lluvia_genil_diaria(fecha_inicio="1995-01-01", fecha_fin="2020-12-31")
    .resample("MS")
    .sum()
    .loc["1995":"2020"]
)

df = pd.DataFrame({"y": caudal_m, "lluvia": lluvia_m}).dropna()
df = df.reset_index().rename(columns={df.index.name or "fecha": "ds"})
df["ds"] = pd.to_datetime(df["ds"])
print(df.head())

## 1 · Split y ajuste

In [ ]:
split = pd.Timestamp("2018-01-01")
train = df[df["ds"] < split].copy()
test = df[df["ds"] >= split].copy()

m = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode="additive",
    changepoint_prior_scale=0.05,
)
m.add_regressor("lluvia")
m.fit(train)

In [ ]:
# Forecast sobre las fechas de test usando la lluvia real
future = pd.concat([train[["ds", "lluvia"]], test[["ds", "lluvia"]]], ignore_index=True)
fc = m.predict(future)
fc.tail(3)[["ds", "yhat", "yhat_lower", "yhat_upper", "trend", "yearly", "lluvia"]]

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(train["ds"], train["y"], color="#1f6f8b", lw=0.8, label="train")
ax.plot(test["ds"], test["y"], color="black", lw=1.4, label="test (real)")
ax.plot(fc["ds"], fc["yhat"], color="#c2410c", lw=1.2, ls="--", label="Prophet yhat")
ax.fill_between(fc["ds"], fc["yhat_lower"], fc["yhat_upper"], color="#c2410c", alpha=0.15)
ax.axvline(split, color="grey", lw=0.5, ls=":")
ax.set_ylabel("Q (m³/s)")
ax.legend()
plt.tight_layout()

## 2 · Componentes

In [ ]:
fig = m.plot_components(fc)
fig.set_size_inches(10, 7)
plt.tight_layout()

**Lectura:**

- *Trend*: detecta changepoints automáticamente (sequía de 2017-18 a menudo aparece como inflexión).
- *Yearly*: ciclo anual claro con pico en marzo-abril.
- *Extra regressors*: el coeficiente positivo de lluvia indica el aporte por unidad de mm.

## 3 · Métricas test

In [ ]:
pred_test = fc[fc["ds"].isin(test["ds"])].set_index("ds")["yhat"]
obs_test = test.set_index("ds")["y"]

rmse = float(np.sqrt(((obs_test - pred_test) ** 2).mean()))
mae = float((obs_test - pred_test).abs().mean())
print(f"Prophet RMSE: {rmse:.3f}   MAE: {mae:.3f}  (test 2018-2020)")

## 4 · Ejercicios

1. **Multiplicativa.** Reajusta con `seasonality_mode='multiplicative'`. ¿Cambia mucho el ajuste? Pista: las amplitudes mensuales del caudal son altas.
2. **Changepoints manuales.** Si tienes hipótesis sobre cuándo se rompió el régimen (sequía 2005, 2017), pásalos vía `changepoints=`.
3. **Más regresores.** Añade `lluvia_lag2` (mes anterior al anterior). ¿Mejora o empeora?
4. **Reto.** Compara la incertidumbre (yhat_upper - yhat_lower) entre Prophet y SARIMAX (notebook 01). ¿Cuál es mejor calibrada respecto al test real?